# AI512 Project: UltraLight VM-UNet on TN3K Thyroid Ultrasound
## Lesion-Scale-Aware Boundary Preservation under Speckle Corruption

This notebook automates the training, evaluation, and complete report deliverables across all 4 experiments:
- **E1**: Original UltraLight VM-UNet Baseline Reproduction
- **E2**: Baseline + Lesion-Scale-Aware Boundary Supervision
- **E3**: Baseline + Corruption-Aware Consistency Training
- **E4**: Full Combined Proposed Framework

> **Smart Checkpointing**: If an experiment is already trained, it is automatically detected and skipped to save time. Set `FORCE_RETRAIN = True` only when you want to retrain from scratch.

### 1. Hardware & GPU Check

In [ ]:
!nvidia-smi

### 2. Fast Dependency Installation (Under 20 Seconds)

In [ ]:
!pip install -q pyyaml scipy scikit-image timm thop pandas matplotlib
print("✅ Core dependencies ready!")

### 3. Sync Latest Code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/YeswanthTejaJakka/ML_UVnet_Project.git"

if not os.path.exists('/content/ML_UVnet'):
    !git clone $REPO_URL /content/ML_UVnet
else:
    %cd /content/ML_UVnet
    !git pull

%cd /content/ML_UVnet

# Verify UltraLight VM-UNet forward pass
import torch
from models import UltraLight_VM_UNet
test_model = UltraLight_VM_UNet().cuda()
dummy = torch.randn(1, 3, 256, 256).cuda()
with torch.no_grad():
    out = test_model(dummy)
print(f"\n✅ SUCCESS: UltraLight VM-UNet verified! Parameters: {sum(p.numel() for p in test_model.parameters()) / 1e6:.4f}M")

### 4. Mount Google Drive and Unpack TN3K Dataset Locally
Unpacks `TN3K.zip` to Colab NVMe scratch disk `/content/tn3k` in under 15 seconds.

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

drive_zip = '/content/drive/MyDrive/TN3K.zip'
drive_folder = '/content/drive/MyDrive/TN3K_extracted/Thyroid Dataset/tn3k'
local_dir = '/content/tn3k'

if not os.path.exists(local_dir):
    if os.path.exists(drive_zip):
        print("Copying TN3K.zip to local /content/...")
        shutil.copy(drive_zip, '/content/TN3K.zip')
        print("Unzipping...")
        !unzip -q /content/TN3K.zip -d /content/
        if not os.path.exists(local_dir) and os.path.exists('/content/Thyroid Dataset/tn3k'):
            !mv "/content/Thyroid Dataset/tn3k" /content/tn3k
        elif not os.path.exists(local_dir) and os.path.exists('/content/TN3K'):
            !mv /content/TN3K /content/tn3k
    elif os.path.exists(drive_folder):
        print("Copying extracted folder from Drive...")
        !cp -r "$drive_folder" /content/tn3k

print("\n--- Dataset Verification ---")
print("TrainVal Images:", len(os.listdir('/content/tn3k/trainval-image')))
print("Test Images:    ", len(os.listdir('/content/tn3k/test-image')))
print("Dataset ready at /content/tn3k")

### 5. Experiment 1 (E1): UltraLight VM-UNet Baseline Reproduction

In [ ]:
%cd /content/ML_UVnet

# Set to True ONLY if you want to retrain from scratch
FORCE_RETRAIN_E1 = False
flag = "--force_retrain" if FORCE_RETRAIN_E1 else ""

!python train.py --config configs/e1_baseline.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e1_baseline $flag

### 6. Experiment 2 (E2): Lesion-Scale-Aware Boundary Supervision

In [ ]:
%cd /content/ML_UVnet

# Set to True ONLY if you want to retrain from scratch
FORCE_RETRAIN_E2 = False
flag = "--force_retrain" if FORCE_RETRAIN_E2 else ""

!python train.py --config configs/e2_boundary.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e2_boundary $flag

### 7. Experiment 3 (E3): Corruption-Aware Consistency Training

In [ ]:
%cd /content/ML_UVnet

# Set to True ONLY if you want to retrain from scratch
FORCE_RETRAIN_E3 = False
flag = "--force_retrain" if FORCE_RETRAIN_E3 else ""

!python train.py --config configs/e3_consistency.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e3_consistency $flag

### 8. Experiment 4 (E4): Full Combined Proposed Framework

In [ ]:
%cd /content/ML_UVnet

# Set to True ONLY if you want to retrain from scratch
FORCE_RETRAIN_E4 = False
flag = "--force_retrain" if FORCE_RETRAIN_E4 else ""

!python train.py --config configs/e4_combined.yaml \
                 --data_root /content/tn3k \
                 --fold 0 \
                 --epochs 50 \
                 --batch_size 16 \
                 --save_dir /content/drive/MyDrive/ML_UVnet_checkpoints/e4_combined $flag

## 📊 DELIVERABLES GENERATION
Executes the full benchmark evaluation on the held-out test set and generates all 8 course deliverables.

In [ ]:
%cd /content/ML_UVnet
!python generate_deliverables.py --checkpoints_root /content/drive/MyDrive/ML_UVnet_checkpoints \
                                --data_root /content/tn3k \
                                --output_dir /content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables

### Deliverable 1: Training Loss & Validation Dice Curves (E1 to E4)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/training_curves_E1_E4.png'
if os.path.exists(img_path):
    plt.figure(figsize=(14, 5))
    plt.imshow(Image.open(img_path))
    plt.axis('off')
    plt.show()

### Deliverable 2: Final Test Comparison Table (E1 vs E2 vs E3 vs E4)

In [ ]:
import pandas as pd
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/final_test_comparison.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)

### Deliverable 3: Lesion-Scale Stratification (Small vs Medium vs Large Nodules)

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/lesion_size_evaluation.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)

### Deliverable 4: Boundary Preservation Metrics (Boundary IoU & HD95)

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/boundary_metrics_summary.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)

### Deliverable 5: Ultrasound Speckle Noise Degradation Benchmark

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/robustness_per_condition.csv'
png_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/speckle_robustness_curve.png'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)
if os.path.exists(png_path):
    plt.figure(figsize=(9, 5))
    plt.imshow(Image.open(png_path))
    plt.axis('off')
    plt.show()

### Deliverable 6: Model Efficiency & Latency Benchmark

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/efficiency_results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)

### Deliverable 7: Qualitative Side-by-Side Visual Comparison (8 Test Cases)

In [ ]:
png_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/qualitative_comparison_8_cases.png'
if os.path.exists(png_path):
    plt.figure(figsize=(18, 16))
    plt.imshow(Image.open(png_path))
    plt.axis('off')
    plt.show()

### Deliverable 8: Final Report Asset Checklist Verification

In [ ]:
from pathlib import Path

root = Path('/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables')
files_to_check = [
    'final_test_comparison.csv',
    'lesion_size_evaluation.csv',
    'boundary_metrics_summary.csv',
    'robustness_per_condition.csv',
    'speckle_robustness_curve.png',
    'efficiency_results.csv',
    'training_curves_E1_E4.png',
    'qualitative_comparison_8_cases.png',
]

print("====================================================")
print("           AI512 REPORT ASSET VERIFICATION          ")
print("====================================================")
all_passed = True
for name in files_to_check:
    p = root / name
    found = p.exists() and p.stat().st_size > 0
    status = "✅ FOUND" if found else "❌ MISSING"
    print(f"{status:10} | {name}")
    if not found:
        all_passed = False

print("----------------------------------------------------")
if all_passed:
    print("🎉 ALL DELIVERABLES SUCCESSFULLY GENERATED AND SAVED!")
else:
    print("⚠️ Some deliverables are missing. Please complete all experiments.")
print("====================================================")